In [ ]:
#数据导入及处理缺失值
#Data import and handling of missing values
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing

# 加载数据
# load data
data = fetch_california_housing()

X = data.data          # 特征值eigenvalue
y = data.target        # 目标值（房价）Target value (house price)

feature_names = data.feature_names

# Transfer to DataFrame（推荐，方便后续操作）
X = pd.DataFrame(X, columns=feature_names)
y = pd.Series(y, name="MedHouseVal")

print(X.head())
print(y.head())
print("Shape of X:", X.shape)

# ===== 处理缺失值 =====
#Handling Missing Values
print("Missing values before cleaning:")
print(X.isnull().sum())

# 删除含缺失值的行（如果有）
# Delete rows with missing values (if any).
X = X.dropna()
y = y.loc[X.index]   # 保持 X 和 y 对齐 Align X and Y

print("Shape after removing missing values:", X.shape)

NameError: name 'X' is not defined

In [ ]:
#Baseline Modal
#Data preprocessing and division of training，validation and test set
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error

# 先从X特征值，y房价 中分出 test 和 train_val两个数据集
# First, separate the "X feature values" and "y house prices" into two datasets: "test" and "train_val".
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=26
)
# X_train_val → 80% 的特征，X_test → 20% 的特征，y_train_val → 80% 的标签，y_test → 20% 的标签
# X_train_val → 80% of the features, X_test → 20% of the features, 
# y_train_val → 80% of the labels, y_test → 20% of the labels

# 从X_train_val, y_train_val里再分 validation（分成X_train, X_val, y_train, y_val）
# From X_train_val and y_train_val, further divide into validation (resulting in X_train, X_val, y_train, y_val)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.1, random_state=26
)#这里的0.1是train_val80%里的10%
#Here, 0.1 refers to the 10% within the "train_val 80%" portion.


# 在标准化前，先保存 raw（未标准化）版本：后面kNN需要用原始经纬度
# Before standardization, save the raw (unstandardized) version first: 
# the kNN algorithm later will require the original latitude and longitude values.
X_train_raw = X_train.copy()
X_val_raw   = X_val.copy()
X_test_raw  = X_test.copy()

# scale (fit only on train) and then convert it back to a DataFrame#标准化（只在训练集）并转回DataFrame
scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns,index=X_train_raw.index)
X_val   = pd.DataFrame(scaler.transform(X_val), columns=X.columns,index=X_val_raw.index)
X_test  = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test_raw.index)
#scaler.transform 标准化 test 数据 means to Standardized test data
#index=X_test_raw.index Pandas 会默认生成新的 0~n 索引 Pandas will automatically generate new indices ranging from 0 to n.
#Add index=...，这样索引不会乱（对齐 y / 后面 debug 更稳）so the index won't be disordered (it will be more stable when aligning y / and then the debug part).

In [ ]:
#寻找最优隐藏层与激活函数
#Search for the optimal hidden layer and activation function
import matplotlib.pyplot as plt
# 1) Define architectures and activation functions
#  定义不同的神经网络结构和激活函数
# ============================================================

# Each tuple represents the number of neurons in hidden layers.
# Example:
# (64,32) means two hidden layers with 64 and 32 neurons.
# 每个元组表示隐藏层结构。
# 例如：
# (64,32) 表示两个隐藏层，分别有 64 和 32 个神经元。
architectures = [
    (32,),
    (64,),
    (64,32),
    (128,32),
    (128,64),
    (128,64,32)
]

# Activation functions to test.
# tanh  → nonlinear symmetric activation
# relu  → widely used deep learning activation
# logistic → sigmoid nonlinear activation
# 要测试的激活函数：
# tanh → 对称的非线性函数
# relu → 深度学习中常用激活函数
# logistic → Sigmoid 非线性激活函数
activations = ['tanh', 'relu', 'logistic']


# ============================================================
# 2) Train models and collect Validation MSE
# 训练模型并记录验证集 MSE
# ============================================================
arch_search_results = []

for arch in architectures:
    for act in activations:

        # Create a neural network model with the current architecture and activation function.
        # 创建一个神经网络模型，使用当前的隐藏层结构和激活函数
        arch_model = MLPRegressor(
            hidden_layer_sizes=arch,
            activation=act,
            solver='adam',
            learning_rate_init=1e-3,
            max_iter=800,
            early_stopping=False,
            random_state=26
        )

        # Train the model on the training dataset.
        # 在训练集上训练模型
        arch_model.fit(X_train, y_train)

        # Predict house prices on the validation set
        # 在验证集上进行预测
        arch_val_pred = arch_model.predict(X_val)
        # Calculate the Mean Squared Error (MSE) on the validation set
        # 计算验证集上的均方误差（MSE）
        arch_val_mse = mean_squared_error(y_val, arch_val_pred) 

        arch_search_results.append({
            'Architecture': str(arch),
            'Activation': act,
            'Validation_MSE': arch_val_mse
        })

        print(f"Architecture: {arch}, Activation: {act}, Validation MSE: {arch_val_mse:.4f}")

# ============================================================
# 3) Convert results to DataFrame
# 将结果转换为表格
# Convert the results list into a pandas DataFrame
# and reshape it using pivot for easier visualization.
# 将实验结果转换为 DataFrame，
# 再使用 pivot 变成适合绘图的格式。

results_df = pd.DataFrame(arch_search_results)

pivot_table = results_df.pivot(
    index='Architecture',
    columns='Activation',
    values='Validation_MSE'
)

# -------------------------------------------------
# Arrange the architecture according to the complexity of the hidden layers
# 按隐藏层的复杂程度进行排列架构
# -------------------------------------------------

order = ['(32,)', '(64,)', '(64, 32)', '(128, 32)','(128, 64)', '(128, 64, 32)']
pivot_table = pivot_table.reindex(order)

print("\n=== Validation MSE Table ===")
print(pivot_table)


# ============================================================
# 4) Plot grouped bar chart
#  绘制分组柱状图
# Create a grouped bar chart showing Validation MSE for each
# architecture and activation combination.
# 绘制分组柱状图，
# 展示不同隐藏层结构和激活函数组合的 Validation MSE。

ax = pivot_table.plot(
    kind='bar',
    figsize=(10,6),
    width=0.8,
    colormap='Set2'
)

plt.ylabel("Validation MSE")
plt.xlabel("Hidden Layer Architecture")
plt.title("Validation MSE for Different Architectures and Activations")
plt.xticks(rotation=0)
plt.legend(title="Activation")




# ------------------------------------------------------------
# Add value labels above bars using patches
# 使用 patches 自动获取每个柱子的位置，避免坐标误差
# ------------------------------------------------------------
    # Place text at the center top of each bar
    # 在每个柱子的顶部中心位置标注数值

# Add value labels above bars
for p in ax.patches:
    height = p.get_height()# 获取柱子的高度（MSE值）Get bar height (MSE value)
    ax.text(
        p.get_x() + p.get_width() / 2,
        height,
        f"{height:.3f}",
        ha='center',
        va='bottom',
        fontsize=9
    )

plt.show()

In [ ]:
# baseline neural network Training

# 1) Combine original train and validation sets
#    合并原来的训练集和验证集
X_train_final_raw_base = pd.concat([X_train_raw, X_val_raw], axis=0)
y_train_final_base = pd.concat([y_train, y_val], axis=0)

# Make sure indices align
# 保证索引一致
X_train_final_raw_base = X_train_final_raw_base.sort_index()
y_train_final_base = y_train_final_base.loc[X_train_final_raw_base.index]

# Refit scaler on the final baseline training set
scaler_base_final = StandardScaler()

X_train_final_base = pd.DataFrame(
    scaler_base_final.fit_transform(X_train_final_raw_base),
    columns=X_train_final_raw_base.columns,
    index=X_train_final_raw_base.index
)

X_test_final_base = pd.DataFrame(
    scaler_base_final.transform(X_test_raw),
    columns=X_test_raw.columns,
    index=X_test_raw.index
)

# model (no internal early stopping, since you already have X_val)
baseline_final_model = MLPRegressor(
    hidden_layer_sizes=(128,32),
    activation='tanh',
    solver='adam',
    learning_rate_init=1e-3,
    max_iter=800,
    early_stopping=False,
    random_state=26
)

baseline_final_model.fit(X_train_final_base, y_train_final_base)

#预测集的预测
#The predictions of the prediction set
baseline_test_pred = baseline_final_model.predict(X_test_final_base)
# 训练集预测
#Training set prediction
baseline_train_pred = baseline_final_model.predict(X_train_final_base)

# 计算训练误差
# Calculate training error
baseline_train_mse = mean_squared_error(y_train_final_base, baseline_train_pred)
baseline_test_mse_final = mean_squared_error(y_test, baseline_test_pred)

print("Final Baseline Train MSE:", baseline_train_mse)#加入是为了判断是不是欠拟合或者过拟合Joining is to determine whether it is underfitting or overfitting.
print("Final Baseline Test MSE:", baseline_test_mse_final)

In [ ]:
# Model 2: Add neighbourhood-based features

from sklearn.neighbors import NearestNeighbors

K = 7  # Number of nearest neighbours used to construct neighbourhood features


#   Fit kNN using training coordinates only, this ensures neighbourhood features are constructed 
#   solely from training data and avoids data leakage.
knn = NearestNeighbors(n_neighbors=K + 1)
knn.fit(X_train_raw[['Latitude', 'Longitude']])

# Align y_train with X_train_raw so neighbour indices correctly map to prices
y_train_aligned = y_train.loc[X_train_raw.index].to_numpy()



# Define a function to compute neighbourhood price statistics
#这部分是用来计算第一个图所需要的数据
def neighbour_summary_feature(query_df, summary="mean", exclude_self=False):
    # Find nearest neighbours based on geographic coordinates
    _, inds = knn.kneighbors(query_df[['Latitude', 'Longitude']])

    values = []

    # Loop through each sample and compute the summary statistic
    for row_i, idxs in enumerate(inds):

        if exclude_self:
            # Remove the sample itself if it appears in neighbours
            idxs = idxs[idxs != row_i]

            # Keep exactly K neighbours
            idxs = idxs[:K]

        else:
            # Validation/test samples are not part of training set
            # so we directly take the first K neighbours
            idxs = idxs[:K]

        # Retrieve neighbour prices
        neigh_prices = y_train_aligned[idxs]

        if summary == "mean":
            values.append(np.mean(neigh_prices))

        elif summary == "median":
            values.append(np.median(neigh_prices))

        elif summary == "min":
            values.append(np.min(neigh_prices))

        elif summary == "max":
            values.append(np.max(neigh_prices))

        elif summary == "range":
            values.append(np.max(neigh_prices) - np.min(neigh_prices))

        else:
            raise ValueError("summary must be one of: mean, median, min, max, range")

    return np.array(values)



# Neighbourhood mean price / 邻域平均房价
neigh_price_mean_train = neighbour_summary_feature(X_train_raw, summary="mean", exclude_self=True)
neigh_price_mean_val   = neighbour_summary_feature(X_val_raw, summary="mean", exclude_self=False)
neigh_price_mean_test  = neighbour_summary_feature(X_test_raw, summary="mean", exclude_self=False)

# Neighbourhood median price / 邻域房价中位数
neigh_price_median_train = neighbour_summary_feature(X_train_raw, summary="median", exclude_self=True)
neigh_price_median_val   = neighbour_summary_feature(X_val_raw, summary="median", exclude_self=False)
neigh_price_median_test  = neighbour_summary_feature(X_test_raw, summary="median", exclude_self=False)

# Neighbourhood minimum price / 邻域最低房价
neigh_price_min_train = neighbour_summary_feature(X_train_raw, summary="min", exclude_self=True)
neigh_price_min_val   = neighbour_summary_feature(X_val_raw, summary="min", exclude_self=False)
neigh_price_min_test  = neighbour_summary_feature(X_test_raw, summary="min", exclude_self=False)

# Neighbourhood maximum price / 邻域最高房价
neigh_price_max_train = neighbour_summary_feature(X_train_raw, summary="max", exclude_self=True)
neigh_price_max_val   = neighbour_summary_feature(X_val_raw, summary="max", exclude_self=False)
neigh_price_max_test  = neighbour_summary_feature(X_test_raw, summary="max", exclude_self=False)

# Neighbourhood price range / 邻域房价极差
neigh_price_range_train = neighbour_summary_feature(X_train_raw, summary="range", exclude_self=True)
neigh_price_range_val   = neighbour_summary_feature(X_val_raw, summary="range", exclude_self=False)
neigh_price_range_test  = neighbour_summary_feature(X_test_raw, summary="range", exclude_self=False)




### 用于计算合并之前的baseline validation MSE的函数
#后面用到的baseline model都是这个，就不用繁琐的每次都写了
scaler_base = StandardScaler()

X_train_base = scaler_base.fit_transform(X_train_raw)
X_val_base = scaler_base.transform(X_val_raw)

baseline_model = MLPRegressor(
    hidden_layer_sizes=(128, 32),
    activation='tanh',
    solver='adam',
    learning_rate_init=1e-3,
    max_iter=800,
    early_stopping=False,
    random_state=26
)

#计算后续需要的baseline validation MSE
baseline_model.fit(X_train_base, y_train)
baseline_val_pred = baseline_model.predict(X_val_base)
baseline_val_mse = mean_squared_error(y_val, baseline_val_pred)

In [ ]:
# ============================================================
# Compare five single neighbourhood-based features
# Each experiment adds only one additional feature
# to the baseline model for comparison
# ============================================================


def run_extra_feature(feature_name, f_train, f_val):
    # ------------------------------------------------------------
    # 1) Copy the original raw feature sets
    #    This prevents modification of the original data
    # ------------------------------------------------------------
    Xtr = X_train_raw.copy()
    Xva = X_val_raw.copy()

    # ------------------------------------------------------------
    # 2) Add the new neighbourhood-based feature
    # ------------------------------------------------------------
    Xtr[feature_name] = f_train
    Xva[feature_name] = f_val

    # ------------------------------------------------------------
    # 3) Re-standardise the dataset after adding the new feature
    #    The scaler is fitted only on the training set
    #    to avoid data leakage
    # ------------------------------------------------------------
    sc = StandardScaler()

    Xtr_s = sc.fit_transform(Xtr)
    Xva_s = sc.transform(Xva)

    # ------------------------------------------------------------
    # 4) Train a neural network model
    #    Use the same architecture as the baseline model
    #    to ensure a fair comparison
    # ------------------------------------------------------------
    

    baseline_model.fit(Xtr_s, y_train)

    # ------------------------------------------------------------
    # 5) Evaluate the model on validation and test sets
    # ------------------------------------------------------------
    baseline_val_pred = baseline_model.predict(Xva_s)

    val_mse = mean_squared_error(y_val, baseline_val_pred)
    return val_mse


# ------------------------------------------------------------
# Run experiments for each neighbourhood feature separately
# ------------------------------------------------------------
single_feature_results = []

# 1) Mean price of K nearest neighbours
val_mse = run_extra_feature(
    "NeighbourPriceMean",
    neigh_price_mean_train, neigh_price_mean_val
)
single_feature_results.append(("NeighbourPriceMean", val_mse))


# 2) Median price of K nearest neighbours
val_mse = run_extra_feature(
    "NeighbourPriceMedian",
    neigh_price_median_train, neigh_price_median_val
)
single_feature_results.append(("NeighbourPriceMedian", val_mse))


# 3) Minimum price among K nearest neighbours
val_mse = run_extra_feature(
    "NeighbourPriceMin",
    neigh_price_min_train, neigh_price_min_val
)
single_feature_results.append(("NeighbourPriceMin", val_mse))


# 4) Maximum price among K nearest neighbours
val_mse = run_extra_feature(
    "NeighbourPriceMax",
    neigh_price_max_train, neigh_price_max_val
)
single_feature_results.append(("NeighbourPriceMax", val_mse))


# 5) Price range among K nearest neighbours
val_mse = run_extra_feature(
    "NeighbourPriceRange",
    neigh_price_range_train, neigh_price_range_val
)
single_feature_results.append(("NeighbourPriceRange", val_mse))


# ------------------------------------------------------------
# Convert results into a DataFrame for easy comparison
# and sort by validation MSE
# ------------------------------------------------------------
single_feature_results_df = pd.DataFrame(single_feature_results, columns=["Feature", "Val MSE"])
single_feature_results_df = single_feature_results_df.sort_values("Val MSE").reset_index(drop=True)

print(single_feature_results_df)

In [ ]:
# ============================================================
# Evaluate combinations of neighbourhood-based features
# This section tests multiple feature combinations
# (1-feature, 2-feature, and 3-feature combinations)
# ============================================================

from itertools import combinations
import pandas as pd


# ------------------------------------------------------------
# Store neighbourhood feature arrays in dictionaries
# Each key represents the feature name
# Each value is the corresponding feature array
# ------------------------------------------------------------
feature_pool_train = {
    "Mean": neigh_price_mean_train,
    "Median": neigh_price_median_train,
    "Min": neigh_price_min_train,
    "Max": neigh_price_max_train,
    "Range": neigh_price_range_train
}

feature_pool_val = {
    "Mean": neigh_price_mean_val,
    "Median": neigh_price_median_val,
    "Min": neigh_price_min_val,
    "Max": neigh_price_max_val,
    "Range": neigh_price_range_val
}



# ------------------------------------------------------------
# Train and evaluate a model with multiple new features
# ------------------------------------------------------------
def run_multi_features(feature_dict_train, feature_dict_val):

    # Copy original datasets to avoid modifying them
    Xtr = X_train_raw.copy()
    Xva = X_val_raw.copy()

    
    for col, arr in feature_dict_train.items():
        Xtr[col] = arr

    for col, arr in feature_dict_val.items():
        Xva[col] = arr

    
    sc = StandardScaler()

    Xtr_s = sc.fit_transform(Xtr)
    Xva_s = sc.transform(Xva)


    baseline_model.fit(Xtr_s, y_train)

    # Evaluate model performance
    baseline_val_pred  = baseline_model.predict(Xva_s)

    return mean_squared_error(y_val, baseline_val_pred)


# ------------------------------------------------------------
# Evaluate different feature combinations
# ------------------------------------------------------------
def evaluate_feature_combinations(
    feature_pool_train,
    feature_pool_val,
    combo_sizes=None,
    selected_combos=None
):

    feature_names = list(feature_pool_train.keys())
    combo_neighbour_results = []

    # Determine which combinations to evaluate
    if selected_combos is not None:
        combos_to_run = selected_combos

    else:
        # If no specific combinations are given,
        # generate combinations automatically
        if combo_sizes is None:
            combo_sizes = list(range(1, len(feature_names) + 1))

        combos_to_run = []

        # Generate combinations of size k
        for k in combo_sizes:
            combos_to_run.extend(combinations(feature_names, k))

    # ------------------------------------------------------------
    # Run experiments for each feature combination
    #
    # 对每个特征组合运行实验
    # ------------------------------------------------------------
    for combo in combos_to_run:

        combo = tuple(combo)

        # Extract the corresponding feature arrays
        feature_dict_train = {name: feature_pool_train[name] for name in combo}
        feature_dict_val   = {name: feature_pool_val[name] for name in combo}

        # Train and evaluate the model
        val_mse = run_multi_features(
            feature_dict_train,
            feature_dict_val,
        )

        # Store experiment results
        combo_neighbour_results.append({
            "combo_name": " + ".join(combo),
            "num_features": len(combo),
            "val_mse": val_mse
        })

    # ------------------------------------------------------------
    # Convert results to DataFrame
    # and sort by number of features and validation MSE
    # ------------------------------------------------------------
    return pd.DataFrame(combo_neighbour_results).sort_values(
        by=["num_features", "val_mse"],
        ascending=[True, True]
    ).reset_index(drop=True)


# ------------------------------------------------------------
# Run the experiment for
# 1-feature, 2-feature, and 3-feature combinations
# ------------------------------------------------------------
combo_results_df = evaluate_feature_combinations(
    feature_pool_train,
    feature_pool_val,
    combo_sizes=[1, 2, 3]
)

# Display the results
# 显示实验结果
display(combo_results_df)

In [ ]:
# ============================================================
# Plot: Validation MSE vs Number of Neighbourhood Features
# ============================================================

import matplotlib.pyplot as plt
import numpy as np


# ------------------------------------------------------------
# Compute summary statistics for each feature count
# (minimum, average, and maximum validation MSE)
# ------------------------------------------------------------
summary_df = (
    combo_results_df
    .groupby("num_features")["val_mse"]   # Group by number of features 
    .agg(["min", "mean", "max"])          # Compute min / mean / max 
    .reset_index()                       # Reset index for plotting 
)

# ------------------------------------------------------------
# Plot scatter points for all feature combinations
# ------------------------------------------------------------
plt.figure(figsize=(10,6))

np.random.seed(26)

plt.scatter(
    combo_results_df["num_features"],   
    combo_results_df["val_mse"],        
    alpha=0.7,
    label="All combinations"
)


# ------------------------------------------------------------
# Plot trend lines
# Show min / average / max performance
# ------------------------------------------------------------
plt.plot(
    summary_df["num_features"],
    summary_df["min"],
    marker="o",
    label="Minimum"
)

plt.plot(
    summary_df["num_features"],
    summary_df["mean"],
    marker="o",
    label="Average"
)

plt.plot(
    summary_df["num_features"],
    summary_df["max"],
    marker="o",
    label="Maximum"
)


# ------------------------------------------------------------
# Plot baseline reference line
# ------------------------------------------------------------
plt.axhline(
    y=baseline_val_mse,
    linestyle="--",
    label="Baseline model"
)


# ------------------------------------------------------------
# Figure formatting
# ------------------------------------------------------------
plt.xlabel("Number of Neighbourhood Features")   # x轴标签
plt.ylabel("Validation MSE")                     # y轴标签
plt.title("Validation MSE for Feature Combinations")  # 图标题

# Show all tested feature counts on x-axis
plt.xticks(sorted(combo_results_df["num_features"].unique()))

plt.legend()                 
plt.grid(True, alpha=0.3)    

plt.show()

In [ ]:
# ============================================================
# Generate results_k_feature_df for Figure 2 (Validation MSE)
# All 2-feature combinations
# ============================================================

from sklearn.neighbors import NearestNeighbors
from itertools import combinations

k_values = [3, 5, 7, 10, 15]

def compute_features_for_k(K):
    knn = NearestNeighbors(n_neighbors=K + 1)
    knn.fit(X_train_raw[['Latitude', 'Longitude']])

    y_train_aligned = y_train.loc[X_train_raw.index].to_numpy()

    # Compute neighbour indices once
    _, idx_train = knn.kneighbors(X_train_raw[['Latitude', 'Longitude']])
    _, idx_val = knn.kneighbors(X_val_raw[['Latitude', 'Longitude']])

    # Remove self for training set
    idx_train = idx_train[:, 1:K+1]
    idx_val = idx_val[:, :K]

    # Convert neighbour indices to price matrices
    train_prices = y_train_aligned[idx_train]
    val_prices = y_train_aligned[idx_val]

    # Vectorised summaries
    neigh_price_train_k = train_prices.mean(axis=1)
    neigh_price_val_k   = val_prices.mean(axis=1)

    neigh_price_median_train_k = np.median(train_prices, axis=1)
    neigh_price_median_val_k   = np.median(val_prices, axis=1)

    neigh_price_min_train_k = train_prices.min(axis=1)
    neigh_price_min_val_k   = val_prices.min(axis=1)

    neigh_price_max_train_k = train_prices.max(axis=1)
    neigh_price_max_val_k   = val_prices.max(axis=1)

    neigh_price_range_train_k = neigh_price_max_train_k - neigh_price_min_train_k
    neigh_price_range_val_k   = neigh_price_max_val_k - neigh_price_min_val_k

    return {
        "Mean":   (neigh_price_train_k, neigh_price_val_k),
        "Median": (neigh_price_median_train_k, neigh_price_median_val_k),
        "Min":    (neigh_price_min_train_k, neigh_price_min_val_k),
        "Max":    (neigh_price_max_train_k, neigh_price_max_val_k),
        "Range":  (neigh_price_range_train_k, neigh_price_range_val_k),
    }

def run_two_features(feature_names, f_train_dict, f_val_dict):
    Xtr = X_train_raw.copy()
    Xva = X_val_raw.copy()

    for name in feature_names:
        Xtr[name] = f_train_dict[name]
        Xva[name] = f_val_dict[name]

    sc = StandardScaler()
    Xtr_s = sc.fit_transform(Xtr)
    Xva_s = sc.transform(Xva)

    baseline_model.fit(Xtr_s, y_train)
    baseline_val_pred = baseline_model.predict(Xva_s)

    return mean_squared_error(y_val, baseline_val_pred)

k_feature_results_df = []

for K in k_values:
    feature_data = compute_features_for_k(K)
    feature_names = list(feature_data.keys())

    for combo in combinations(feature_names, 2):
        f_train_dict = {name: feature_data[name][0] for name in combo}
        f_val_dict   = {name: feature_data[name][1] for name in combo}

        val_mse = run_two_features(combo, f_train_dict, f_val_dict)

        k_feature_results_df.append({
            "k": K,
            "combo": " + ".join(combo),
            "val_mse": val_mse
        })

k_feature_results_df = pd.DataFrame(k_feature_results_df)

In [ ]:
# ============================================================
# Figure 2: Validation MSE vs Number of Neighbours (k)
# All 2-feature combinations
# ============================================================

plt.figure(figsize=(10, 6))


# ------------------------------------------------------------
# Plot validation MSE curves for each feature combination
# ------------------------------------------------------------
for combo_name in k_feature_results_df["combo"].unique():

    # Select rows corresponding to the current feature combination
    subset = k_feature_results_df[
        k_feature_results_df["combo"] == combo_name].sort_values("k")   

    # Plot validation MSE as a function of k
    plt.plot(
        subset["k"],        
        subset["val_mse"],  
        marker="o",
        label=combo_name
    )


# ------------------------------------------------------------
# Plot baseline model performance as a reference line
# ------------------------------------------------------------

plt.axhline(
    y=baseline_val_mse,
    linestyle="--",
    label="Baseline model"
)


# ------------------------------------------------------------
# Figure formatting
# ------------------------------------------------------------

plt.xlabel("Number of Neighbours (k)")
plt.ylabel("Validation MSE")


plt.title("Validation MSE vs Number of Neighbours (Two-Feature Combinations)")

# Ensure all tested k values appear on the x-axis
plt.xticks(sorted(k_feature_results_df["k"].unique()))

# Place legend outside the plot to avoid overlapping
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")

# Add grid for readability
plt.grid(True, alpha=0.3)

# Display the figure
plt.show()

In [ ]:
# ============================================================
# 1) Combine original train and validation sets
# ============================================================
X_train_final_raw = pd.concat([X_train_raw, X_val_raw], axis=0)
y_train_final = pd.concat([y_train, y_val], axis=0)

# Make sure indices align
X_train_final_raw = X_train_final_raw.sort_index()
y_train_final = y_train_final.loc[X_train_final_raw.index]

# ============================================================
# 2) Fit kNN on the new final training set
# ============================================================
best_K = 7
knn_final = NearestNeighbors(n_neighbors=best_K + 1)
knn_final.fit(X_train_final_raw[['Latitude', 'Longitude']])

y_train_final_aligned = y_train_final.to_numpy()

# ============================================================
# 3) Define function to compute neighbourhood summary features
# ============================================================
def neighbour_summary_feature_final(query_df, summary="mean", exclude_self=False):
    _, inds = knn_final.kneighbors(query_df[['Latitude', 'Longitude']])
    values = []

    for row_i, idxs in enumerate(inds):
        if exclude_self:
            idxs = idxs[idxs != row_i]
            idxs = idxs[:best_K]
        else:
            idxs = idxs[:best_K]

        neigh_prices = y_train_final_aligned[idxs]

        if summary == "mean":
            values.append(np.mean(neigh_prices))
        elif summary == "median":
            values.append(np.median(neigh_prices))
        elif summary == "min":
            values.append(np.min(neigh_prices))
        elif summary == "max":
            values.append(np.max(neigh_prices))
        elif summary == "range":
            values.append(np.max(neigh_prices) - np.min(neigh_prices))
        else:
            raise ValueError("summary must be one of: mean, median, min, max, range")

    return np.array(values)

# ============================================================
# 4) Compute only the selected best features
# ============================================================
feature_map = {
    "Mean": "mean",
    "Median": "median",
    "Min": "min",
    "Max": "max",
    "Range": "range"
}

best_features = ["Max", "Median"]   # 输入最终选出来的组合

train_final_feature_values = {}
test_feature_values = {}

for feat in best_features:
    summary_name = feature_map[feat]
    train_final_feature_values[feat] = neighbour_summary_feature_final(
        X_train_final_raw, summary=summary_name, exclude_self=True
    )
    test_feature_values[feat] = neighbour_summary_feature_final(
        X_test_raw, summary=summary_name, exclude_self=False
    )


# ============================================================
# 6) Final enhanced model
# ============================================================
X_train_enh_final_raw = X_train_final_raw.copy()
X_test_enh_final_raw = X_test_raw.copy()

for feat in best_features:
    col_name = f"NeighbourPrice{feat}"
    X_train_enh_final_raw[col_name] = train_final_feature_values[feat]
    X_test_enh_final_raw[col_name] = test_feature_values[feat]

scaler_enh_final = StandardScaler()

X_train_enh_final = pd.DataFrame(
    scaler_enh_final.fit_transform(X_train_enh_final_raw),
    columns=X_train_enh_final_raw.columns,
    index=X_train_enh_final_raw.index
)

X_test_enh_final = pd.DataFrame(
    scaler_enh_final.transform(X_test_enh_final_raw),
    columns=X_test_enh_final_raw.columns,
    index=X_test_enh_final_raw.index
)

model_enh_final = MLPRegressor(
    hidden_layer_sizes=(128, 32),
    activation='tanh',
    solver='adam',
    learning_rate_init=1e-3,
    max_iter=800,
    early_stopping=False,
    random_state=26
)

model_enh_final.fit(X_train_enh_final, y_train_final)
test_pred_enh_final = model_enh_final.predict(X_test_enh_final)
enhanced_test_mse_final = mean_squared_error(y_test, test_pred_enh_final)

# ============================================================
# 7) Final comparison
# ============================================================
print("Final Baseline Test MSE:", baseline_test_mse_final)
print("Final Enhanced Test MSE:", enhanced_test_mse_final)
print("Delta Test MSE (Enhanced - Baseline):", enhanced_test_mse_final - baseline_test_mse_final)

In [ ]:
# ============================================================
# Overfitting / Underfitting Check for Model 1 and Model 2
# 过拟合/欠拟合检查 — 模型1（基线）与模型2（含邻域特征）
# ============================================================

# ── Diagnosis helper / 诊断辅助函数 ──────────────────────────
def diagnose(model_name, train_mse, val_mse, threshold_ratio=1.5):
    """
    Print an overfitting/underfitting diagnosis based on Val/Train MSE ratio.
    根据验证集与训练集MSE比值，打印过拟合/欠拟合诊断结果。
    """
    # Compute ratio of validation MSE to training MSE
    # 计算验证集MSE与训练集MSE的比值
    ratio = val_mse / train_mse if train_mse > 0 else float('inf')

    print(f"\n{'='*55}")
    print(f"  {model_name}")
    print(f"{'='*55}")
    print(f"  Train MSE : {train_mse:,.2f}")
    print(f"  Val   MSE : {val_mse:,.2f}")
    print(f"  Val/Train : {ratio:.3f}")

    # Diagnosis rules / 诊断规则:
    # Train MSE very high             → Underfitting / 训练误差极高 → 欠拟合
    # Val/Train ratio > threshold     → Overfitting  / 比值过大     → 过拟合
    # Otherwise                       → Good fit     / 否则         → 拟合良好
    if train_mse > 1e10:
        print("  ⚠ Diagnosis : UNDERFITTING — training error is very high")
        print("  ⚠ 诊断结果  : 欠拟合 — 训练误差过高，模型未能有效学习")
    elif ratio > threshold_ratio:
        print("  ⚠ Diagnosis : OVERFITTING  — val MSE is much larger than train MSE")
        print("  ⚠ 诊断结果  : 过拟合 — 验证集误差远大于训练集误差")
    else:
        print("  ✓ Diagnosis : GOOD FIT     — train and val MSE are close")
        print("  ✓ 诊断结果  : 拟合良好 — 训练集与验证集误差接近")
    print(f"{'='*55}")


# ============================================================
# MODEL 1 — Baseline
# 模型1 — 基线模型
# Reuse scaler_base, X_train_base, X_val_base, baseline_model
# 直接复用你原代码中已定义的 scaler_base 和 baseline_model
# ============================================================

# baseline_model was already fitted on X_train_base earlier
# baseline_model 在前面已经用 X_train_base 训练完毕，直接预测即可
train_mse_m1 = mean_squared_error(y_train, baseline_model.predict(X_train_base))
val_mse_m1   = mean_squared_error(y_val,   baseline_model.predict(X_val_base))

diagnose("Model 1 — Baseline", train_mse_m1, val_mse_m1)


# ============================================================
# MODEL 2 — With all five neighbourhood features
# 模型2 — 包含全部五个邻域统计特征
# ============================================================

# Copy raw feature sets to avoid modifying original data
# 复制原始特征集，避免污染原有变量
X_train_m2 = X_train_raw.copy()
X_val_m2   = X_val_raw.copy()

# Append all five neighbourhood price statistics as new input features
# 将五个邻域房价统计值追加为新的输入特征列
X_train_m2['NeighbourPriceMean']   = neigh_price_mean_train
X_train_m2['NeighbourPriceMedian'] = neigh_price_median_train
X_train_m2['NeighbourPriceMin']    = neigh_price_min_train
X_train_m2['NeighbourPriceMax']    = neigh_price_max_train
X_train_m2['NeighbourPriceRange']  = neigh_price_range_train

X_val_m2['NeighbourPriceMean']     = neigh_price_mean_val
X_val_m2['NeighbourPriceMedian']   = neigh_price_median_val
X_val_m2['NeighbourPriceMin']      = neigh_price_min_val
X_val_m2['NeighbourPriceMax']      = neigh_price_max_val
X_val_m2['NeighbourPriceRange']    = neigh_price_range_val

# Re-standardise after adding new features
# Scaler is fitted only on training set to avoid data leakage
# 加入新特征后重新标准化，scaler仅在训练集上拟合以避免数据泄露
scaler_m2    = StandardScaler()
X_train_m2_s = scaler_m2.fit_transform(X_train_m2)
X_val_m2_s   = scaler_m2.transform(X_val_m2)

# Train Model 2 using the same architecture as the baseline for fair comparison
# 使用与基线模型完全相同的架构训练模型2，保证对比公平
model2 = MLPRegressor(
    hidden_layer_sizes=(128, 32),
    activation='tanh',
    solver='adam',
    learning_rate_init=1e-3,
    max_iter=800,
    early_stopping=False,
    random_state=26
)
model2.fit(X_train_m2_s, y_train)

# Compute MSE on both training and validation sets
# 分别在训练集和验证集上计算MSE
train_mse_m2 = mean_squared_error(y_train, model2.predict(X_train_m2_s))
val_mse_m2   = mean_squared_error(y_val,   model2.predict(X_val_m2_s))

diagnose("Model 2 — With Neighbourhood Features", train_mse_m2, val_mse_m2)


# ============================================================
# VISUALISATION 1 — Bar chart: Train vs Val MSE
# 可视化1 — 柱状图：训练集与验证集MSE对比
# ============================================================

labels     = ['Model 1\n(Baseline)', 'Model 2\n(+ Neighbourhood)']
train_mses = [train_mse_m1, train_mse_m2]
val_mses   = [val_mse_m1,   val_mse_m2]

x     = np.arange(len(labels))  # X-axis positions / x轴位置
width = 0.35                     # Bar width / 柱子宽度

fig, ax = plt.subplots(figsize=(8, 5))

# Plot train and val MSE bars side by side
# 并排绘制训练集和验证集MSE柱子
bars1 = ax.bar(x - width/2, train_mses, width, label='Train MSE', color='steelblue',  alpha=0.85)
bars2 = ax.bar(x + width/2, val_mses,   width, label='Val MSE',   color='darkorange', alpha=0.85)

ax.set_ylabel('MSE')
ax.set_title('Train vs Validation MSE — Model 1 and Model 2\n训练集与验证集MSE对比')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()

# Annotate each bar with its exact value
# 在柱子顶部标注具体数值
ax.bar_label(bars1, fmt='{:,.0f}', padding=3, fontsize=8)
ax.bar_label(bars2, fmt='{:,.0f}', padding=3, fontsize=8)

plt.tight_layout()
plt.show()


# ============================================================
# VISUALISATION 2 — Learning curves (MSE vs Epoch)
# 可视化2 — 学习曲线（MSE随训练轮数变化）
#
# Uses warm_start=True to train one epoch at a time and
# record train/val MSE after each epoch.
# 使用 warm_start=True 逐epoch训练，每轮结束后记录MSE。
#
# How to interpret / 学习曲线解读:
#   Both lines fall and converge         → Good fit    两线下降并收敛    → 拟合良好
#   Val rises while Train keeps falling  → Overfitting 验证线上升训练线降 → 过拟合
#   Both lines stay high                 → Underfitting 两线居高不下      → 欠拟合
# ============================================================

def collect_loss_curve(X_tr, y_tr, X_va, y_va, max_iter=800):
    """
    Train epoch-by-epoch and record train/val MSE at each step.
    逐epoch训练并在每轮结束后记录训练集与验证集的MSE。
    """
    # warm_start=True lets the model continue from where it left off
    # warm_start=True 使模型在每次 fit 时从上次结束处继续训练，而非重新初始化
    model = MLPRegressor(
        hidden_layer_sizes=(128, 32),
        activation='tanh',
        solver='adam',
        learning_rate_init=1e-3,
        max_iter=1,         # Train exactly one epoch per call / 每次只训练一个epoch
        warm_start=True,    # Retain weights between calls / 保留上次训练权重
        early_stopping=False,
        random_state=26
    )
    train_losses, val_losses = [], []

    for _ in range(max_iter):
        model.fit(X_tr, y_tr)  # Continue training for one more epoch / 继续训练一个epoch

        # Record MSE after this epoch / 记录本epoch结束后的MSE
        train_losses.append(mean_squared_error(y_tr, model.predict(X_tr)))
        val_losses.append(mean_squared_error(y_va,   model.predict(X_va)))

    return train_losses, val_losses


print("\nCollecting learning curves (this may take a moment)…")
print("正在收集学习曲线数据（可能需要一点时间）…")

# Collect learning curves for both models
# 分别收集两个模型的学习曲线数据
lc_train_m1, lc_val_m1 = collect_loss_curve(X_train_base,  y_train, X_val_base,  y_val)
lc_train_m2, lc_val_m2 = collect_loss_curve(X_train_m2_s,  y_train, X_val_m2_s,  y_val)

# Plot side-by-side learning curves
# 并排绘制两个模型的学习曲线
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, lc_tr, lc_va, title in zip(
    axes,
    [lc_train_m1, lc_train_m2],
    [lc_val_m1,   lc_val_m2],
    ['Model 1 — Baseline', 'Model 2 — + Neighbourhood Features']
):
    epochs = range(1, len(lc_tr) + 1)

    # Solid line = Train MSE, Dashed line = Val MSE
    # 实线 = 训练集MSE，虚线 = 验证集MSE
    ax.plot(epochs, lc_tr, label='Train MSE', color='steelblue')
    ax.plot(epochs, lc_va, label='Val MSE',   color='darkorange', linestyle='--')

    ax.set_title(title)
    ax.set_xlabel('Epoch / 训练轮数')
    ax.set_ylabel('MSE')
    ax.legend()
    ax.grid(True, linestyle=':', alpha=0.6)

plt.suptitle(
    'Learning Curves — Overfitting / Underfitting Check\n学习曲线 — 过拟合/欠拟合检查',
    fontsize=13
)
plt.tight_layout()
plt.show()